<a href="https://colab.research.google.com/github/dataprogpy/code-samples/blob/main/starter_files/09_time_series_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Time Series Analysis

In [ ]:
%pip install statsforecast

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import datetime as dt
from functools import partial


import polars as pl
import pandas as pd
import yfinance as yf

import altair as alt
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox

from statsforecast import StatsForecast
# from pmdarima import auto_arima
# from pmdarima import model_selection

from statsforecast.models import (
    HoltWinters,
    CrostonClassic as Croston,
    HistoricAverage,
    DynamicOptimizedTheta as DOT,
    SeasonalNaive,
    AutoARIMA,
    GARCH,
    _TS
)

from statsforecast.utils import AirPassengersDF

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import (
    mse,
    rmse,
    mae,
    mape,
    mase,
    smape
    )

import utilsforecast.losses as ufl

## Data Wrangling and Exploratory Data Analysis

In [ ]:
df_missing = pl.DataFrame({
    "date": [dt.date(2024, 1, 1), dt.date(2024, 1, 2), dt.date(2024, 1, 3)],
    "value": [10, None, 15]
})

df = (
    df_missing
    .with_columns(
        pl.col("value")
        .fill_null(strategy="forward")
        .alias("value_forward")
        )
    .with_columns(
        pl.col("value")
        .fill_null(strategy="backward")
        .alias("value_backward")
    )
    .with_columns(
        pl.col("value")
        .fill_null(strategy="mean")
        .alias("value_mean")
    )
)
df

In [ ]:
# Example DataFrame with daily data
df_daily = pl.DataFrame({
    "date": pl.date_range(dt.date(2024, 1, 1), dt.date(2024, 2, 15), "1d", eager=True),
    "sales": range(46)
})

# Downsample to monthly average sales
df_monthly = df_daily.group_by_dynamic(
    "date", every="1mo"
).agg(
    pl.col("sales").mean().alias("average_sales")
)
display(df_monthly)

In [ ]:
# Define the ticker symbol and date range
ticker = 'GOOG'
start_date = '2019-01-01'
end_date = '2025-06-01'

# Download the data using yfinance
df = yf.download(ticker, start=start_date, end=end_date)

# Display the first few rows of the DataFrame
display(df.head())
df_pl = pl.from_pandas(df, include_index=True )
df_pl.columns

In [ ]:
def colname(col: str):
  if col.startswith('('):
        return ( col
          .lstrip('(')
          .rstrip(')')
          .split(",")[0]
          .strip("'")
          .lower()
        )
  return col.lower()

print([colname(col) for col in df_pl.columns])

df_pl = df_pl.rename(colname)
df_pl.head()

In [ ]:
daily = alt.Chart(df_pl).mark_line().encode(
    x='date',
    y='close',
    tooltip=['date', 'close']
).properties(
    title="Daily Prices - GOOG",
    width=800,
    height=400
)

df_monthly = df_pl.group_by_dynamic("date", every="1mo", period="1mo").agg(pl.col("close").mean())

monthly = alt.Chart(df_monthly).mark_line().encode(
    x='date',
    y='close',
    tooltip=['date', 'close']
).properties(
    title="Down Sampled - (Monthly)",
    width=800,
    height=400
)
display(daily)
display(monthly)


### Filtering data based on temporal properties

In [ ]:
alt.Chart(
    df_pl.filter(
         pl.col("date").is_between(
             dt.datetime(2020, 1, 1),
             dt.datetime(2022, 12, 31)
             )
    )
    ).mark_line().encode(
    x='date',
    y='close',
    tooltip=['date', 'close']
).properties(
    width=800,
    height=400
)

In [ ]:
# Calculate 30-day and 90-day moving averages
df_with_ma = df_pl.with_columns(
    pl.col("close").rolling_mean(window_size=30).alias("ma_30_day"),
    pl.col("close").rolling_mean(window_size=90).alias("ma_90_day")
)
# Plot the daily closing price
base = alt.Chart(df_with_ma).encode(x='date:T')

closing_price = base.mark_line().encode(
    y=alt.Y('close:Q', title='Closing Price (USD)')
).properties(
    title='Google (GOOG) Daily Stock Price',
    width=800,
    height=400,
)

# Create layers for the moving averages
ma_30 = base.mark_line(color='orange').encode(
    y='ma_30_day:Q'
)

ma_90 = base.mark_line(color='red').encode(
    y='ma_90_day:Q'
)

# Combine the original price plot with the moving average plots
(closing_price + ma_30 + ma_90)

## Visual Exploration of Trend and Seasonlity

In [ ]:
df = pd.read_csv('https://datasets-nixtla.s3.amazonaws.com/air-passengers.csv', parse_dates=['ds'])
df.head()

In [ ]:
df_ap = pl.from_pandas(df)
df_ap.head()

In [ ]:

df_ap = df_ap.with_columns(
    pl.col("y").rolling_mean(window_size=12).alias("ma_12_month"),
    pl.col("ds").dt.month().alias("season"),
).with_columns(
    (pl.col("y")/pl.col("ma_12_month")).alias("detrended")
)

df_ap.tail()

In [ ]:
base = alt.Chart(df_ap)

full = base.mark_line().encode(
    alt.X('ds:T'),
    alt.Y('y:Q'),
    tooltip=['ds', 'y']
).properties(
    width=800,
    height=400
)

ma_12_month = full.mark_line(color='orange').encode(
    alt.Y('ma_12_month:Q'),
)

detrended = full.mark_line(color='red').encode(
    alt.Y('detrended:Q').scale(zero=False)
)

seasonlity = (
    full
    .mark_boxplot()
    .encode(
        alt.X('season:O'),
        alt.Y('detrended:Q').scale(zero=False),
    )
)

display(full + ma_12_month )
display(detrended)
display(seasonlity)

In [ ]:
df.head()

In [ ]:
df = df.set_index('ds')
decomposition = sm.tsa.seasonal_decompose(df['y'], model='multiplicative', period=12)

# Plot the decomposed components
fig = decomposition.plot()
fig.set_size_inches(10, 8)
plt.show()

In [ ]:
df_diff = df.copy()
df_diff['y_diff'] = df_diff['y'].diff()

df_diff.plot()



In [ ]:
differenced_series = df['y'].diff().dropna()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

plot_acf(differenced_series, ax=ax1, lags=40)
plot_pacf(differenced_series, ax=ax2, lags=40)

plt.show()

In [ ]:
model = sm.tsa.ARIMA(differenced_series, order=(1, 0, 1))
results = model.fit()

# Print the model summary
print(results.summary())

# Generate forecasts
forecast_steps = 24
forecast = results.forecast(steps=forecast_steps)

# Plot the original series and the forecast
plt.figure(figsize=(12, 6))
plt.plot(differenced_series, label='Observed')
plt.plot(forecast, label='Forecast', color='red')
plt.title('ARIMA Forecast')
plt.legend()
plt.show()

## Modeling with `StatsForecast`

In [ ]:
df = AirPassengersDF
sf = StatsForecast(
    models=[AutoARIMA(season_length = 12)],
    freq='MS',
)
sf.fit(df)
forecast_df = sf.predict(h=12, level=[90])
forecast_df.tail()

In [ ]:
sf.plot(df, forecast_df, level=[90])

### Model Selection

In [ ]:
# Create a list of models and instantiation parameters
models = [
    HoltWinters(),
    Croston(),
    SeasonalNaive(season_length=12),
    HistoricAverage(),
    DOT(season_length=12)
]

# Instantiate StatsForecast class as sf
sf = StatsForecast(
    models=models,
    freq='MS',
    n_jobs=-1,
    fallback_model=SeasonalNaive(season_length=7),
    verbose=True,
)

# Fit & Predict
forecasts_df = sf.forecast(df=df, h=48, level=[90])
forecasts_df.head()

In [ ]:
sf.plot(df,forecasts_df)

In [ ]:
# Plot to unique_ids and some selected models
sf.plot(df, forecasts_df, models=["DynamicOptimizedTheta"], unique_ids=[1.0], level=[90])

### Cross Validation & Backtesting

In [ ]:
cv_df = sf.cross_validation(
    df=df,
    h=12,
    step_size=12,
    n_windows=2
)
cv_df.head()

In [ ]:
def evaluate_cv(df, metric):
    models = df.columns.drop(['unique_id', 'ds', 'y', 'cutoff']).tolist()
    evals = metric(df, models=models)
    evals['best_model'] = evals[models].idxmin(axis=1)
    return evals

evaluation_df = evaluate_cv(cv_df, mse)
display(evaluation_df.head())
evaluation_df['best_model'].value_counts()

## End-to-End Time-Series Modeling Demo
Fitting a GARCH Model using `StatsForecast`

In [ ]:
ticker = '^GSPC'
period1 = dt.datetime(2015, 1, 1)
period2 = dt.datetime(2023, 9, 22)
interval = '1d' # 1d, 1m

SP_500 = yf.download(ticker, start=period1, end=period2, interval=interval, progress=False)
SP_500 = SP_500.reset_index()

SP_500.head()

In [ ]:
df=SP_500[["Date","Close"]]
df["unique_id"]="1"
df.columns=["ds", "y", "unique_id"]
display(df.head())
StatsForecast.plot(df)

### Augmented Dickey-Fuller Stationarity Test

In [ ]:
def Augmented_Dickey_Fuller_Test_func(series , column_name):
    print (f'Dickey-Fuller test results for columns: {column_name}')
    dftest = adfuller(series, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','No Lags Used','Number of observations used'])
    for key,value in dftest[4].items():
       dfoutput['Critical Value (%s)'%key] = value
    print (dfoutput)
    if dftest[1] <= 0.05:
        print("Conclusion:====>")
        print("Reject the null hypothesis")
        print("The data is stationary")
    else:
        print("Conclusion:====>")
        print("The null hypothesis cannot be rejected")
        print("The data is not stationary")

In [ ]:
Augmented_Dickey_Fuller_Test_func(df["y"],'S&P500')

In [ ]:
df['return'] = 100 * df["y"].pct_change()
df.dropna(inplace=True, how='any')
df['sq_return'] = df["return"].mul(df["return"])

display(df.head())

base = alt.Chart(df).mark_line().encode(
    x='ds',
    # y='return',
).properties(
    width=800,
    height=400,
    # title="SP500 Return Chart"
)
return_chart = base.encode(
    y='return'
).properties(
    title="SP500 Return Chart"
)
sq_return_chart = base.encode(
    y='sq_return'
).properties(
    title="SP500 Squared Return Chart"
)

(return_chart | sq_return_chart).properties(
    title="SP500 Return and Squared Return Chart"
)
# df.head()

### Ljung-Box Autocorrelation Test

In [ ]:
ljung_res = acorr_ljungbox(df["return"], lags= 40, boxpierce=True)

# ln_pvalue < 0.05 ? reject null hypotheses i.e. no autocorrelation
ljung_res.head()

In [ ]:
df=df[["ds","unique_id","return"]]
df.columns=["ds", "unique_id", "y"]
train = df[df.ds<='2023-05-31'] # Let's forecast the last 30 days
test = df[df.ds>'2023-05-31']
train.shape, test.shape

### Cross Validation & Model Selection

In [ ]:
season_length = 7 # Dayly data
horizon = len(test) # number of predictions biasadj=True, include_drift=True,

models = [GARCH(1,1),
          GARCH(1,2),
          GARCH(2,2),
          GARCH(2,1),
          GARCH(3,1),
          GARCH(3,2),
          GARCH(3,3),
          GARCH(1,3),
          GARCH(2,3)]

sf = StatsForecast(
    models=models,
    freq='C', # custom business day frequency
)

crossvalidation_df = sf.cross_validation(df=train,
                                         h=horizon,
                                         step_size=6,
                                         n_windows=5)

crossvalidation_df

In [ ]:
evals = evaluate(crossvalidation_df.drop(columns='cutoff'), metrics=[rmse], agg_fn='mean')
evals

In [ ]:
season_length = 7 # Dayly data
horizon = len(test) # number of predictions biasadj=True, include_drift=True,

models = [GARCH(1,1)]

sf = StatsForecast(models=models,
                   freq='C', # custom business day frequency
                  )
sf.fit(df=train)

StatsForecast(models=[GARCH(1,1)], freq='C')

result=sf.fitted_[0,0].model_
display(result)

residual=pd.DataFrame(result.get("actual_residuals"), columns=["residual Model"])
display(residual)

In [ ]:
Y_hat = sf.forecast(df=train, h=horizon, fitted=True, level=[95])
display(Y_hat.head())
values=sf.forecast_fitted_values()
display(values.head())
sf.plot(train, Y_hat.merge(test), max_insample_length=200)

In [ ]:
forecast_df = sf.predict(h=horizon, level=[80,95])
sf.plot(train, test.merge(forecast_df), level=[80, 95], max_insample_length=200)

### Model Evaluation

In [ ]:
evaluate(
    test.merge(Y_hat),
    metrics=[mae, mape, partial(mase, seasonality=season_length), rmse, smape],
    train_df=train,
)

## Next Steps

- Try time series modeling on your project dataset.
- Read up on different models and their hyperparameters.
- Understand different evaluation metrics used in rating model performance.

## References

- [StatsForecast - Quick Start](https://nixtlaverse.nixtla.io/statsforecast/docs/getting-started/getting_started_short.html)
- [Pandas - Time Series Frequency/Offset Alias](https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#offset-aliases)
- [Polars - Time Series Frequency/Offset Alias](https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.dt.offset_by.html#polars.Series.dt.offset_by)
- [Cross Validation using `StatsForecast`](https://nixtlaverse.nixtla.io/statsforecast/docs/getting-started/getting_started_complete.html#evaluate-the-model%E2%80%99s-performance)
- [AutoARIMA Model](https://nixtlaverse.nixtla.io/statsforecast/docs/models/autoarima.html)
- [HoltWinters Model](https://nixtlaverse.nixtla.io/statsforecast/docs/models/holtwinters.html)
- [CrostonClassic Model](https://nixtlaverse.nixtla.io/statsforecast/docs/models/crostonclassic.html)
- [SeasonalNaive Model](https://nixtlaverse.nixtla.io/statsforecast/src/core/models.html#seasonalnaive-2)
- [Dynamic Optimized Theta Model](https://nixtlaverse.nixtla.io/statsforecast/docs/models/dynamicoptimizedtheta.html)